# Graph Structure Learning for Collective Intelligence

> **Core idea:** N agents with noisy local observations. No graph given. Learn which agents should communicate — end-to-end.

## What this notebook does
1. Clones your repo and installs dependencies
2. Runs a quick sanity check on synthetic data (SBM graph)
3. Loads METR-LA traffic data (if available) or continues on synthetic
4. Trains GSLNet and logs to W&B
5. **The killer result:** visualizes how well the learned graph recovers the true graph — without ever seeing it during training
6. Runs the ablation: Random vs kNN vs Learned


In [ ]:
# ── Cell 1: Clone repo & install ──────────────────────────────────────────────
import os

REPO_URL = "https://github.com/YOUR_USERNAME/gsl-collective-intelligence"  # ← change this
REPO_DIR = "gsl-collective-intelligence"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, 'src')

import torch
import yaml
import matplotlib.pyplot as plt
import numpy as np

from gsl.data import SyntheticGraphDataset, MetrLADataset, get_dataloaders
from gsl.model import GSLNet
from gsl.loss import JointLoss
from gsl.train import train
from gsl.evaluate import (
    graph_recovery_auroc, task_metrics, avg_degree,
    plot_adjacency_comparison, plot_learned_neighborhoods
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ── Cell 3: (Optional) W&B login ──────────────────────────────────────────────
# Highly recommended — free tier is enough.
# If you skip this, training still works, just no logging.
try:
    import wandb
    wandb.login()  # paste your API key when prompted
    wandb.init(project='gsl-collective-intelligence', config={'stage': 'colab'})
    print('W&B initialized')
except Exception as e:
    print(f'W&B skipped: {e}')

In [ ]:
# ── Cell 4: Sanity check on Synthetic (SBM) ───────────────────────────────────
# Ground truth graph IS known here → perfect for debugging

with open('configs/synthetic.yaml') as f:
    cfg = yaml.safe_load(f)

dataset = SyntheticGraphDataset(
    num_nodes=cfg['num_nodes'],
    in_features=cfg['in_features'],
    num_samples=500,
    seed=cfg['seed']
)

print(f'Dataset: {len(dataset)} samples, {cfg["num_nodes"]} nodes, {cfg["in_features"]} features')
print(f'Ground truth graph density: {dataset.A_true.mean():.3f}')

# Quick peek at the ground truth adjacency
plt.figure(figsize=(5, 4))
plt.imshow(dataset.A_true.numpy(), cmap='Blues', aspect='auto')
plt.title('Ground Truth Adjacency (SBM)')
plt.colorbar()
plt.show()

In [ ]:
# ── Cell 5: Train on synthetic ─────────────────────────────────────────────────
train_loader, val_loader, test_loader = get_dataloaders(
    dataset, cfg['train_ratio'], cfg['val_ratio'], cfg['batch_size'], cfg['seed']
)

model = GSLNet(
    in_features=cfg['in_features'],
    hidden_dim=cfg['hidden_dim'],
    num_classes=5,
    gnn_layers=cfg['gnn_layers'],
    metric='cosine',
    sparsify='top_k',
    top_k=cfg['top_k'],
    task='node'
)

criterion = JointLoss(
    task='classification',
    sparsity_lambda=cfg['sparsity_lambda']
)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'])

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

model = train(model, train_loader, val_loader, criterion, optimizer, cfg, device)

In [ ]:
# ── Cell 6: The killer result — graph recovery ────────────────────────────────
# Model never saw the graph during training.
# Does the learned adjacency recover the ground truth?

model.eval()
model = model.to(device)

# Collect learned adjacency matrices over test set, then average
A_accum = torch.zeros(cfg['num_nodes'], cfg['num_nodes'])
count = 0

with torch.no_grad():
    for x, _ in test_loader:
        x = x.to(device)
        _, A_batch = model(x)    # [B, N, N]
        A_accum += A_batch.mean(0).cpu()
        count += 1

A_learned = A_accum / count

auroc = graph_recovery_auroc(A_learned, dataset.A_true)
degree = avg_degree(A_learned)

print(f'Graph Recovery AUROC: {auroc:.4f}  (0.5 = random chance, 1.0 = perfect)')
print(f'Average learned degree: {degree:.1f} (top_k={cfg["top_k"]})')

plot_adjacency_comparison(A_learned, dataset.A_true,
                           title=f'Graph Recovery AUROC = {auroc:.3f}',
                           save_path='results/adjacency_recovery.png')

In [ ]:
# ── Cell 7: Task accuracy ──────────────────────────────────────────────────────
all_logits, all_targets = [], []
model.eval()
with torch.no_grad():
    for x, y in test_loader:
        logits, _ = model(x.to(device))
        # logits: [B, N, C] — flatten to [B*N, C]
        all_logits.append(logits.view(-1, logits.size(-1)).cpu())
        all_targets.append(y.view(-1).cpu())

all_logits = torch.cat(all_logits)
all_targets = torch.cat(all_targets)
metrics = task_metrics(all_logits, all_targets)
print(f'Test Accuracy: {metrics["accuracy"]:.4f}')
print(f'Test F1 (macro): {metrics["f1_macro"]:.4f}')

In [ ]:
# ── Cell 8: Interpretability — learned neighborhoods ──────────────────────────
# Pick a few nodes and visualize which neighbors the model learned to attend to.
# This is the slide that impresses.

plot_learned_neighborhoods(
    A_learned,
    node_ids=[0, 10, 25],
    top_k=5,
    save_path='results/neighborhoods.png'
)

# Check: are highly-weighted neighbors in the same community?
labels_np = dataset.labels.numpy()
A_np = A_learned.numpy()

print('\nCommunity coherence of learned neighborhoods:')
for node in [0, 10, 25]:
    w = A_np[node].copy(); w[node] = 0
    top5 = np.argsort(w)[-5:][::-1]
    same = sum(labels_np[n] == labels_np[node] for n in top5)
    print(f'  Node {node} (community {labels_np[node]}): {same}/5 top neighbors in same community')

In [ ]:
# ── Cell 9: Load METR-LA (if you have it) ─────────────────────────────────────
# Mount Google Drive first, then point to the files.

# from google.colab import drive
# drive.mount('/content/drive')

# Uncomment and set path after mounting:
# DATA_PATH = '/content/drive/MyDrive/metrla/metr-la.h5'
# ADJ_PATH  = '/content/drive/MyDrive/metrla/adj_mx.pkl'

# dataset_metrla = MetrLADataset(data_path=DATA_PATH, adj_path=ADJ_PATH)
# print(f'METR-LA: {len(dataset_metrla)} samples, A_true shape: {dataset_metrla.A_true.shape}')

print('Uncomment the lines above after mounting Drive and downloading METR-LA.')
print('Download: https://drive.google.com/drive/folders/10FOTa6HXPqX8Pf5WRoRwcFnW9BrNZEIX')

In [ ]:
# ── Cell 10: Run full ablation ─────────────────────────────────────────────────
# Random graph vs kNN vs Learned — produces the bar chart
!python experiments/run_ablation.py --config configs/synthetic.yaml

In [ ]:
# ── Cell 11: Save checkpoint to Google Drive ───────────────────────────────────
import shutil

# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy('results/best_model.pt', '/content/drive/MyDrive/gsl_checkpoints/best_model.pt')
# print('Saved to Drive.')

print('Unmount block: uncomment after mounting Drive.')